In [ ]:
# ============================================
# 03_CNN.ipynb - VERSI CEPAT (CPU OPTIMIZED)
# ============================================

import os
import sys
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

# ============================================
# SETUP
# ============================================
warnings.filterwarnings('ignore')
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU

import tensorflow as tf

# CPU Optimization
tf.config.threading.set_intra_op_parallelism_threads(8)
tf.config.threading.set_inter_op_parallelism_threads(4)

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print("="*60)
print("🧠 CNN FAST TRAINING (CPU OPTIMIZED)")
print("="*60)

# ============================================
# CONFIGURATION - DIPERCEPAT
# ============================================

project_root = Path.cwd().parent if Path.cwd().name in ['notebooks', 'notebook'] else Path.cwd()
processed_dir = project_root / "dataset" / "processed" / "cnn_mobilenet"
model_dir = project_root / "models" / "cnn"
result_dir = project_root / "results" / "cnn"

model_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'img_size': (128, 128),  # ← LEBIH KECIL = LEBIH CEPAT
    'batch_size': 16,
    'epochs': 20,            # ← LEBIH SEDIKIT
    'learning_rate': 0.001,
    'early_stopping_patience': 5,  # ← STOP LEBIH CEPAT
    'reduce_lr_patience': 3,
    'class_names': ['awake', 'drowsy']
}

print(f"\n[Fast Configuration]")
for key, value in CONFIG.items():
    print(f"   - {key}: {value}")
print(f"\n   ⚡ Estimasi waktu: ~45-60 menit (dari 2-3 jam)")

# ============================================
# LOAD DATA
# ============================================

print("\n[Loading Data]")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_path = processed_dir / "train"
valid_path = processed_dir / "valid"
test_path = processed_dir / "test"

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
    class_mode='binary',
    classes=CONFIG['class_names'],
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    valid_path,
    target_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
    class_mode='binary',
    classes=CONFIG['class_names'],
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=CONFIG['img_size'],
    batch_size=CONFIG['batch_size'],
    class_mode='binary',
    classes=CONFIG['class_names'],
    shuffle=False
)

print(f"\n   - Train: {train_generator.samples}")
print(f"   - Valid: {validation_generator.samples}")
print(f"   - Test: {test_generator.samples}")

# ============================================
# BUILD SIMPLE CNN
# ============================================

print("\n[Building Lightweight CNN]")

def build_lightweight_cnn():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        
        # Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 4
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        
        # Dense
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_lightweight_cnn()

model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model compiled")
model.summary()

# ============================================
# TRAINING CEPAT
# ============================================

print("\n[Training Model - FAST MODE]")
print("="*60)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'], 
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=CONFIG['reduce_lr_patience'],
                      min_lr=1e-7, verbose=1),
    ModelCheckpoint(model_dir / 'best_model.keras', monitor='val_accuracy', 
                    save_best_only=True, verbose=1),
]

start_time = time.time()

history = model.fit(
    train_generator,
    epochs=CONFIG['epochs'],
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n⏱️ Training time: {training_time/60:.2f} minutes")

# ============================================
# EVALUASI
# ============================================

print("\n[Evaluating Model]")

best_model = keras.models.load_model(model_dir / 'best_model.keras')
test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=0)

print(f"\n📊 Test Results:")
print(f"   - Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   - Loss: {test_loss:.4f}")

# ============================================
# PLOT
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True)

plt.savefig(result_dir / 'training_history.png', dpi=150)
plt.show()                          

print("\n" + "="*60)
print("✅ CNN FAST TRAINING COMPLETE!")
print(f"   Accuracy: {test_accuracy*100:.2f}%")
print(f"   Time: {training_time/60:.2f} minutes")
print("="*60)